In [ ]:
%cd /kaggle/working
!git clone --branch feature/issue-5-densenet https://github.com/hai291/AI66B-Computer-vision.git
%cd /kaggle/working/AI66B-Computer-vision
%pip install -q pydicom python-gdcm


In [ ]:
from pathlib import Path
import torch

roots = [p.parent for p in Path("/kaggle/input").rglob("train_label_coordinates.csv")
         if (p.parent / "train_images").is_dir() and (p.parent / "train.csv").is_file()]
assert len(roots) == 1, "Gan dung mot dataset RSNA co train_images vao Add Input"
DATA_ROOT = roots[0]
print("Data:", DATA_ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Chua bat GPU")
assert torch.cuda.is_available(), "Vao Settings > Accelerator de bat GPU truoc khi train"


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pydicom
from train_densenet import read_samples, SpineCrops

samples = read_samples(DATA_ROOT, seed=42, max_studies=20)
row = samples.iloc[0]
original = pydicom.dcmread(DATA_ROOT / row["path"]).pixel_array
image, label = SpineCrops(samples.iloc[:1], DATA_ROOT, 128, 224)[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(original, cmap="gray")
axes[0].add_patch(Rectangle((round(row.x) - 64, round(row.y) - 64),
                            128, 128, fill=False, edgecolor="red", linewidth=2))
axes[0].set_title("Anh goc va vung crop")
axes[1].imshow(image[0], cmap="gray", vmin=-1, vmax=1)
axes[1].set_title(f"{row.condition} | {row.level} | {row.severity}")
plt.show()


In [ ]:
# Chay thu 20 ca. Ket qua nay chi de kiem tra pipeline.
!python train_densenet.py --data-root "{DATA_ROOT}" --model densenet --epochs 1 --max-studies 20 --output /kaggle/working/runs/smoke


In [ ]:
# Train CNN baseline tren toan bo crop co nhan.
!python train_densenet.py --data-root "{DATA_ROOT}" --model baseline --epochs 5 --seed 42 --batch-size 16 --output /kaggle/working/runs/baseline


In [ ]:
# Fine-tune DenseNet121 voi cung split va cau hinh train.
!python train_densenet.py --data-root "{DATA_ROOT}" --model densenet --epochs 5 --seed 42 --batch-size 16 --output /kaggle/working/runs/densenet


In [ ]:
# Doc checkpoint de evaluate lai, khong train them.
!python train_densenet.py --data-root "{DATA_ROOT}" --evaluate /kaggle/working/runs/densenet


In [ ]:
import json
import pandas as pd

output = Path("/kaggle/working/runs")
baseline_samples = pd.read_csv(output / "baseline/samples.csv")
densenet_samples = pd.read_csv(output / "densenet/samples.csv")
pd.testing.assert_frame_equal(baseline_samples, densenet_samples)
rows = []
for name in ["baseline", "densenet"]:
    metrics = json.loads((output / name / "metrics.json").read_text())
    config = json.loads((output / name / "config.json").read_text())
    rows.append({"model": name, "pretrained": config["pretrained"],
                 "epoch": metrics["epoch"], "log_loss": metrics["log_loss"],
                 "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"]})
comparison = pd.DataFrame(rows)
comparison.to_csv(output / "comparison.csv", index=False)
display(comparison)
